# M54 · Coulomb stress transfer: stress on a receiver fault, two Okada codes, and what aftershocks can test

**Week 7 · about 10 h** · before this: M04, M50, M52

> **한국어 요약** 이 모듈에서는 한 지진의 단층 미끄러짐이 주변 암석의 응력을 얼마나 바꾸는지, 그리고 그 변화가 다른 단층(수용 단층)을 미끄러지기 쉽게 하는지 어렵게 하는지를 나타내는 쿨롱 파괴 응력 변화(ΔCFS)를 직접 계산합니다. 먼저 응력 텐서를 수용 단층의 미끄러짐 방향 전단 응력과 수직 응력으로 분해하는 함수를 Elastic_stresses_py와 똑같은 정의로 작성하고, 그 프로그램의 입력 형식(인자 순서와 단층의 기준점)에 맞추어 단층을 적는 함수를 만듭니다. 이어서 pyrocko의 Okada 코드로 직사각형 단층 주변의 응력을 계산하고 Elastic_stresses_py와 비교하며, 지진 모멘트와 응력 강하로 단층 크기를 구해 경험식(Wells와 Coppersmith 1994)과 견주어 봅니다. 2019년 상주 지진(ML 3.9)에서는 여진 거리에서의 응력 변화를 조석 응력과 숫자로 비교하고, 2016년 경주 지진에서는 USGS와 ISC에서 받은 메커니즘으로 전진과 본진을 단층으로 만들어 Elastic_stresses_py를 그대로 다시 실행한 뒤 마찰 계수 하나를 바꾸어 실행하고, KMA 여진이 양(+)의 ΔCFS 영역에 놓이는 비율을 회전 귀무 검정과 비교합니다. 마지막으로 순천 지진들의 단층 크기, 시간 분포, 위치 간격을 카탈로그에서 숫자로 확인하고, 그 숫자로 쿨롱 검정이 그곳에서 무엇을 가려낼 수 있는지 판단합니다.

**You will**

- Resolve a stress tensor onto a receiver fault (shear in the rake direction, normal stress with unclamping positive, pore pressure, Coulomb change) exactly as Elastic_stresses_py defines it, and check it against that program
- Describe one rectangular fault for two programs whose input conventions you look up in their code (the point of the rectangle, the order of the angles), and measure how closely Elastic_stresses_py and pyrocko's Okada code agree on its stress
- Estimate the size and slip of a fault from its moment and a stress drop, compare them with an empirical scaling relation, and compare the Coulomb change at aftershock distances with tidal stresses in numbers
- Run Elastic_stresses_py for the 2016 Gyeongju foreshock and mainshock, rerun it unchanged, change the friction and explain what changed, and test the fraction of aftershocks in positive areas against a rotation null whose chance level you measure
- Measure the fault size, the timing and the location step of the small Suncheon earthquakes from the catalog, and judge from those numbers what a Coulomb test could resolve there

**New words:** Coulomb failure stress change (ΔCFS; 쿨롱 파괴 응력 변화), stress tensor (응력 텐서), traction (견인력 (트랙션)), shear stress (전단 응력), normal stress (수직 응력), effective friction coefficient (유효 마찰 계수), Skempton coefficient (B; 스켐프턴 계수), pore pressure (공극압) · [all terms](../../../docs/glossary.md)

**Optional reading:** King, Stein & Lin (1994), the whole paper (the Coulomb failure criterion, the stress lobes around a strike-slip fault and the role of friction); Stein (1999), the sections on stress shadows and on triggering; Harris (1998), the introduction; Okada (1992), the abstract and the description of the rectangular source; Shearer (2019), the chapter on earthquakes (stress drop and the circular crack)

Log any AI help in `work/ai_log/M54.md` ([rules](../../../docs/ai_policy.md)); pass criteria: [criteria/M54.yaml](../../../.sgtl/criteria/M54.yaml).

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
FRICTION = 0.4                            # effective friction coefficient on receiver faults (King et al. 1994)
SKEMPTON = 0.0                            # Skempton coefficient B (0: no pore-pressure change)
SHEAR_MODULUS_PA = 30e9                   # mu of the uniform half-space (Pa)
LAME1_PA = 30e9                           # lambda (Lame's first parameter, Pa); lambda = mu gives Poisson's ratio 0.25
STRESS_DROP_MPA = 3.0                     # Parts 4 and 6: stress drop that sets fault size from moment (MPa)
TIDAL_KPA = 1.0                           # Part 4: order of the tidal stress amplitude in the crust (kPa)
SANGJU_EVENT_ID = "2019009762"            # the ML 3.9 Sangju earthquake of 21 July 2019 in the KMA catalog
SANGJU_PLANE = (288.0, 89.0, 7.0)         # strike, dip, rake; KIGAM (2021), report KEMS-2019-062, plane A
SANGJU_MW = 3.8                           # moment magnitude of the same report
SANGJU_DEPTH_KM = 12.8                    # focal depth of the same report (km)
GYEONGJU_MAIN_ID = "2016000291"           # KMA event ids: the ML 5.8 mainshock of 12 September 2016
GYEONGJU_FORE_ID = "2016000276"           # and the ML 5.1 foreshock 48 minutes earlier
USGS_MAIN_EVENT = "us10006p1f"            # USGS ComCat id of the mainshock
USGS_FORE_EVENT = "us10006p12"            # USGS ComCat id of the foreshock
FORE_WINDOW_UTC = ("2016-09-12T10:44:00", "2016-09-12T10:45:30")   # ISC search window for the foreshock
STRIKE_RANGE = (0.0, 60.0)                # Part 6: keep the nodal plane whose strike lies in this range (degrees)
AFTERSHOCK_DAYS = 365                     # Part 6: aftershocks up to this many days after the mainshock
EXCLUDE_KM = 2.0                          # Part 6: leave out points closer than this to either fault rectangle
PROFILE_DEPTH_KM = 15.0                   # Part 6: depth of Elastic_stresses_py's map of the Coulomb change
N_ROTATIONS = 200                         # Part 6: turns of the aftershock cloud in the rotation test
SEED = 54                                 # Part 6: seed of the rotation test
CHANGED_FRICTION = 0.0                    # Part 6 (R3): the one value you change in the third run
FIGURE_DIR = "figures"                    # your figure files go here, next to this notebook
RESULT_DIR = "results"                    # your result files (runs, tables) go here

In [ ]:
import sgtl_tutorial as st
S = st.init("M54")

import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from sgtl_tutorial._paths import short_path

Path(FIGURE_DIR).mkdir(exist_ok=True)
Path(RESULT_DIR).mkdir(exist_ok=True)


def show_figure(figure, filename):
    """Save and embed the same static figure, including in a headless executed notebook."""
    output = Path(FIGURE_DIR) / filename
    figure.savefig(output, dpi=150)
    display(Image(filename=str(output)))
    plt.close(figure)


def zero_boundary(axis, x, y, values):
    """Mark a zero contour only when the displayed field actually crosses zero."""
    finite = np.asarray(values)[np.isfinite(values)]
    if finite.size and finite.min() < 0.0 < finite.max():
        contour = axis.contour(x, y, values, levels=[0.0], colors="0.2", linewidths=0.7, linestyles=":")
        axis.clabel(contour, fmt={0.0: "0 kPa"}, fontsize=7)


def patch_projection(axis, patch, label, color, depth=False):
    """Draw geometry supplied by the production reader, without recomputing a fault input line."""
    east, north, _, _ = patch.get_fault_four_corners()
    vertical = [patch.top, patch.top, patch.bottom, patch.bottom, patch.top] if depth else north
    axis.plot(east, vertical, color=color, lw=1.5, label=label)
    center = patch.get_fault_center()
    axis.plot(center[0], center[2] if depth else center[1], "o", color=color, ms=4)

## How this notebook works

The same five steps as in the earlier modules, for every task: read the **Contract**, write a **prediction**,
replace `raise NotImplementedError(...)` under `# YOUR CODE HERE` with your code, run the check cell, and answer
the **Explain** prompt in 3-5 sentences of your own. `st.hint("M54-T1", 1)` shows a first hint; hints 2 and 3
unlock after failed checks. Tasks 1-5 are checked on made-up faults, tensors and maps whose answers are known
from how they were built or from another program (`st.reroll("M54-T1")` draws new ones); task 6 reruns your own
functions on your own Gyeongju files and never prints a value of its own for them.

The module climbs a ladder in three steps. **R1**: write the core functions (resolving a stress on a fault,
writing a fault for a program, stress around a rectangle, fault size, a rotation test) and test them on made-up
cases. **R2**: compare two independent programs on the same fault and apply your functions to real earthquakes
(Sangju 2019, Gyeongju 2016). **R3**: run the production tool, Elastic_stresses_py (Materna and Wong 2023), for
Gyeongju: once, again unchanged, and once with one value changed.

Before Part 1, run `bin/sgtl-install coulomb` in a server terminal, read its plan, then repeat with `--apply`.
Elastic_stresses_py supplies the independent checks as well as the production runs; keep the course kernel.
The first Gyeongju mechanism query needs internet access and saves its result in
`results/gyeongju_mechanisms.json` for later reruns. Part 0 asks for two of your own checked functions.

You may ask an AI tool to explain an error message or a concept. It may not write the task functions, your
predictions or your Explain answers ([docs/ai_policy.md](../../../docs/ai_policy.md)); log AI conversations in
`work/ai_log/M54.md`.

## Part 0 · Stress transfer in one paragraph, and two functions you already wrote (about 20 min)

When a fault slips, the rock around it is pushed and pulled: the **stress** (force per area) changes everywhere
nearby. The fault that slipped is the **source fault** (glossary `source_fault`); a fault on which we ask "did this
change bring it closer to failure?" is a **receiver fault** (glossary `receiver_fault`). The usual measure is the
**Coulomb failure stress change** (glossary `coulomb_failure_stress_change`, ΔCFS): the change of shear stress in the
direction the receiver would slip, plus a friction coefficient times the change of normal stress (positive when the
fault is unclamped). Positive ΔCFS brings a receiver closer to failure; negative ΔCFS moves it away, a **stress
shadow** (glossary `stress_shadow`). King et al. (1994), Harris (1998) and Stein (1999) introduced and reviewed the
idea; Toda et al. (2005) used it to forecast seismicity; Hardebeck et al. (1998) showed how hard a fair test is.

You need two functions from earlier modules. From M50 (task M50-T1), your `fault_vectors(strike, dip, rake)`: the
unit **fault normal** (glossary `fault_normal`, pointing up into the hanging wall) and the unit **slip vector**
(glossary `slip_vector`, the motion of the hanging wall), in the **north-east-down** frame (glossary
`north_east_down`). From M52 (task M52-T2), your `magnitude_moment(mw=None, m0_nm=None)`. Paste both, as they are
in your notebooks, into the next cell and run it.

In [ ]:
# Paste your fault_vectors of module M50 and your magnitude_moment of module M52 below this line.

In [ ]:
for name, task in (("fault_vectors", "M50-T1 of module M50"), ("magnitude_moment", "M52-T2 of module M52")):
    if not callable(globals().get(name)):
        raise NameError(f"{name} not found: paste your function into the cell of Part 0 and run it "
                        f"(finish task {task} first).")
n_test, d_test = fault_vectors(0.0, 90.0, 180.0)
print("your M50 vectors of a vertical fault striking north, rake 180: normal",
      (np.round(n_test, 3) + 0.0).tolist(), " slip", (np.round(d_test, 3) + 0.0).tolist(), "(north, east, down)")
print(f"your M52 function: Mw 3.0 -> M0 = {magnitude_moment(mw=3.0):.4g} N m")

## Part 1 · Resolving a stress change onto a receiver fault (about 60 min, then task 1)

A **stress tensor** (glossary `stress_tensor`) σ is a symmetric 3 x 3 matrix: the force per area on any plane
through a point. The force per area on one plane with unit normal **n** is the **traction** (glossary `traction`)
**t** = σ **n**. Its part along **n** is the **normal stress** (glossary `normal_stress`) σ_n = **t**·**n**; its
part along a direction **d** inside the plane is a **shear stress** (glossary `shear_stress`) τ = **t**·**d**.

Elastic_stresses_py (Materna and Wong 2023), the program of Parts 2 and 6, uses these conventions (read them in
its file `conversion_math.py`, function `get_coulomb_stresses_internal`; the next cell prints where it is):

- the tensor's rows and columns are **east, north, up** (x east, y north, z up), in Pa;
- tension is positive, so a positive normal stress change **unclamps** the receiver (pulls its sides apart);
- the plane normal points from the footwall into the hanging wall; the shear is taken in the **rake direction**
  of the receiver, the direction its hanging wall would slip;
- with fluids in the rock, a change of the **mean stress** (glossary `mean_stress`, the trace of σ divided by 3)
  changes the **pore pressure** (glossary `pore_pressure`) by an amount set by the **Skempton coefficient**
  (glossary `skempton_coefficient`) B, and the **effective** normal stress change is the dry one plus that pore
  change (Beeler et al. 2000); with B = 0 they are equal;
- ΔCFS = shear + μ' × effective normal, with μ' the **effective friction** coefficient (glossary
  `effective_friction`), often 0.4 (King et al. 1994);
- the results come out in **kPa** (1 kPa = 1000 Pa; 100 kPa = 1 bar).

Your M50 vectors are in north, east, down; the tensor here is in east, north, up. One more thing to read in that
file: the order in which `get_coulomb_stresses` takes the receiver's angles.

In [ ]:
import elastic_stresses_py.PyCoulomb.conversion_math as esp_math
print("Elastic_stresses_py's conversion math:", short_path(esp_math.__file__))

### M54-T1 · A stress change resolved onto a receiver fault · core · about 60 min

**Goal:** write `coulomb_change(stress_enu_pa, strike_deg, dip_deg, rake_deg, friction, skempton=0.0)`: the shear,
normal and Coulomb stress changes on a receiver fault, exactly as Elastic_stresses_py defines them.

**Given:** Part 1, your `fault_vectors` of M50 (Part 0) and numpy. Not allowed: Elastic_stresses_py,
tectonic-utils, cutde, pyrocko, MTUQ and the course package (the check compares with Elastic_stresses_py).

**Contract:**
- Inputs: `stress_enu_pa`, one stress-change tensor (3 x 3) or a stack of n tensors (n x 3 x 3), in Pa, rows and
  columns east, north, up, tension positive; the receiver's `strike_deg`, `dip_deg`, `rake_deg` (degrees, in this
  order); `friction` (the effective friction coefficient); `skempton` (B, 0 to 1).
- Output: a tuple `(shear_kpa, normal_kpa, coulomb_kpa)`: floats for one tensor, numpy arrays of length n for a
  stack. `shear_kpa`: shear stress change in the receiver's rake direction; `normal_kpa`: the effective normal
  stress change (unclamping positive, pore-pressure change included); `coulomb_kpa` = shear + friction × normal.
  All in kPa.
- Examples (an invented tensor, in Pa: [[120e3, -40e3, 25e3], [-40e3, -60e3, 10e3], [25e3, 10e3, 30e3]]):
  receiver 30/60/-45, friction 0.4, skempton 0 gives (61.4616, 104.1506, 103.1219); the same with skempton 0.5
  gives (61.4616, 89.1506, 97.1219); receiver 200/90/180, friction 0.6, skempton 0 gives (-27.2091, 124.6555,
  47.5842).

**Done when** `st.check("M54-T1", func=coulomb_change)` passes on 40 made-up tensors and receivers (all strikes,
dips from 0 to 90, all rakes, some stacks), against Elastic_stresses_py's `get_coulomb_stresses`.

**Stuck?** `st.hint("M54-T1", 1)`

In [ ]:
# Before you code: a vertical receiver striking north (dip 90) and the tensor with only sigma_xy = sigma_yx =
# +100 kPa (east-north shear): which rake, 0 or 180, gets a positive shear stress, and what is its normal stress?
# Which sign does a tensor of pure compression (all three diagonal elements -100 kPa) give to the normal stress,
# and does it change the shear?
st.predict("M54-T1", """

""")

In [ ]:
def coulomb_change(stress_enu_pa, strike_deg, dip_deg, rake_deg, friction, skempton=0.0):
    """Shear, effective normal and Coulomb stress changes (kPa) on a receiver, as Elastic_stresses_py defines them.

    stress_enu_pa : (3, 3) or (n, 3, 3) stress change in Pa, axes east, north, up, tension positive
    strike_deg, dip_deg, rake_deg : the receiver (degrees, in this order)
    friction      : effective friction coefficient
    skempton      : Skempton coefficient B (0 to 1)
    returns       : (shear_kpa, normal_kpa, coulomb_kpa), floats for one tensor, arrays (n,) for a stack
    """
    # Step 1: the receiver's normal and slip vector from your M50 fault_vectors, turned into east, north, up.
    # Step 2: the traction on the receiver, and its parts along the slip vector and along the normal.
    # Step 3: the pore-pressure change from the mean stress, in Elastic_stresses_py's convention.
    # Step 4: kPa, and Coulomb = shear + friction x normal.
    # YOUR CODE HERE
    raise NotImplementedError("M54-T1: write your code here")

In [ ]:
st.check("M54-T1", func=coulomb_change)

Your function on the examples of the Contract and on the two tensors of the prediction:

In [ ]:
ex_tau = np.array([[120e3, -40e3, 25e3], [-40e3, -60e3, 10e3], [25e3, 10e3, 30e3]])
for rec, fr, b in (((30.0, 60.0, -45.0), 0.4, 0.0), ((30.0, 60.0, -45.0), 0.4, 0.5), ((200.0, 90.0, 180.0), 0.6, 0.0)):
    print(f"receiver {rec}, friction {fr}, skempton {b}:", np.round(coulomb_change(ex_tau, *rec, fr, b), 4))
xy_shear = np.zeros((3, 3))
xy_shear[0, 1] = xy_shear[1, 0] = 100e3
squeeze = -100e3 * np.eye(3)
for rake in (0.0, 180.0):
    print(f"east-north shear, receiver 0/90/{rake:g}:", np.round(coulomb_change(xy_shear, 0.0, 90.0, rake, 0.4), 4))
print("pure compression, receiver 0/90/0:", np.round(coulomb_change(squeeze, 0.0, 90.0, 0.0, 0.4), 4))
print("pure compression, skempton 0.5:   ", np.round(coulomb_change(squeeze, 0.0, 90.0, 0.0, 0.4, 0.5), 4))

**Read the signs as a picture.** The matrix is the prescribed east-north shear input. In map view, the vertical
receiver is the north-south line; the arrows are your checked M50 normal and slip directions, drawn as unit
directions rather than stress magnitudes. The bars come from your T1 function: shear, friction times effective
normal stress, and their Coulomb result. A negative normal contribution clamps the receiver in this convention.

**Interpret:** follow a tensor column to the traction on an east-facing plane. Why does reversing rake reverse
the shear contribution, while compression changes the normal contribution? Which part moves when B changes?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.0, 3.8), constrained_layout=True)
matrix = axes[0].imshow(xy_shear / 1000.0, cmap="RdBu_r", vmin=-100.0, vmax=100.0)
axes[0].set(xticks=range(3), yticks=range(3), xticklabels=["E", "N", "U"], yticklabels=["E", "N", "U"],
            xlabel="Tensor column", ylabel="Tensor row", title="Prescribed stress change")
for row in range(3):
    for col in range(3):
        axes[0].text(col, row, f"{xy_shear[row, col] / 1000.0:g}", ha="center", va="center")
fig.colorbar(matrix, ax=axes[0], label="Stress change (kPa)", shrink=0.8)
axes[1].plot([0.0, 0.0], [-1.2, 1.2], color="k", lw=3, label="Vertical receiver")
for rake, color in ((0.0, "tab:blue"), (180.0, "tab:orange")):
    normal_ned, slip_ned = fault_vectors(0.0, 90.0, rake)
    axes[1].quiver(0.0, 0.0, slip_ned[1], slip_ned[0], angles="xy", scale_units="xy", scale=1,
                   color=color, width=0.025, label=f"Slip, rake {rake:g}°")
axes[1].quiver(0.0, 0.0, normal_ned[1], normal_ned[0], angles="xy", scale_units="xy", scale=1,
               color="tab:green", width=0.018, label="Normal")
axes[1].set(xlim=(-1.3, 1.3), ylim=(-1.3, 1.3), xlabel="East (unit direction)",
            ylabel="North (unit direction)", title="Receiver orientation", aspect="equal")
axes[1].legend(loc="upper left", fontsize=7)
examples = [("Shear\nrake 0", xy_shear, 0.0, 0.0), ("Shear\nrake 180", xy_shear, 180.0, 0.0),
            ("Compression\nB = 0", squeeze, 0.0, 0.0), ("Compression\nB = 0.5", squeeze, 0.0, 0.5)]
budgets = np.array([coulomb_change(tensor, 0.0, 90.0, rake, 0.4, b) for _, tensor, rake, b in examples])
for offset, column, label, color in ((-0.25, budgets[:, 0], "Shear", "tab:blue"),
                                   (0.0, 0.4 * budgets[:, 1], "Friction × effective normal", "tab:orange"),
                                   (0.25, budgets[:, 2], "Coulomb", "0.25")):
    axes[2].bar(np.arange(4) + offset, column, width=0.24, label=label, color=color)
axes[2].axhline(0.0, color="k", lw=0.7)
axes[2].set(xticks=range(4), xticklabels=[row[0] for row in examples], ylabel="Stress change (kPa)",
            title="Your resolved stress budget")
axes[2].legend(fontsize=7, loc="lower left")
show_figure(fig, "M54_traction_and_signs.png")

**Explain (3-5 sentences):** Use the printed lines. Which rake got the positive shear under the east-north shear,
and does that match your prediction? What did pure compression give for the normal stress with B = 0 and with B =
0.5, and why does the pore-pressure change move it the way it did? In which order does Elastic_stresses_py's own
function take strike, dip and rake, and what would a caller who passes them in the order of your function get?

## Part 2 · One fault, written for Elastic_stresses_py (about 40 min, then task 2)

Elastic_stresses_py reads its faults from a text file (its "intxt" format). One rectangular fault with uniform
slip is one `Source_Patch:` line with nine numbers: three angles, the length and the width (km), a longitude, a
latitude and a depth (km), and the slip (m). Two things are not obvious, and you must find them in its reader
(`inputs_object/io_intxt.py`, functions `read_source_line_slip_convention` and `compute_params_for_slip_source`):
the order of the three angles, and **which point of the rectangle** the longitude, latitude and depth describe.
Catalogs and papers give you something else: a hypocentre (where the rupture started) or a centroid (the middle of
the slip). Here you are given the **centre** of the rectangle: half way along its length and half way down its
width.

You can always ask the program what it read: its reader builds a fault object, and `get_fault_center()` gives that
fault's centre in km east and north of a reference point (`zerolon`, `zerolat`) and its depth. The next cell shows
how to read a line back; try `get_fault_center()` on it yourself when you study the reader.

In [ ]:
from elastic_stresses_py.PyCoulomb.inputs_object import io_intxt
print("Elastic_stresses_py's reader of the intxt format:", short_path(io_intxt.__file__))
# How to read a line back (an invented line, not of any real fault):
demo = io_intxt.get_source_patch("Source_Patch: 10 20 80 3.0 2.0 127.0 36.0 5.0 0.2", 127.0, 36.0)
print("read back: length", round(demo.L, 4), "km, width", round(demo.W, 4), "km; the fault object's type:",
      type(demo).__name__)

### M54-T2 · A fault in Elastic_stresses_py's input format · core · about 45 min

**Goal:** write `patch_line(strike_deg, dip_deg, rake_deg, length_km, width_km, center_lon, center_lat,
center_depth_km, slip_m)`: the `Source_Patch:` line of Elastic_stresses_py that describes the rectangle whose
centre you are given.

**Given:** Part 2 and Elastic_stresses_py's reader (read its code; do not call it in your function). numpy and
pyproj are allowed. Not allowed: Elastic_stresses_py, tectonic-utils and the course package.

**Contract:**
- Inputs: strike, dip and rake of the fault (degrees; the dip more than 0 and less than 90: Elastic_stresses_py
  refuses a dip of exactly 90 and its own writer puts 89.99 for a vertical fault); its length along strike and
  width down dip (km); the longitude, latitude and depth (km) of its centre; the slip (m, positive; its direction
  is the rake).
- Output: one line of text, `"Source_Patch: "` followed by the nine numbers in the order the reader expects. Write
  longitude and latitude with at least 6 decimals and the other numbers with at least 6 significant digits.
- The check reads your line with Elastic_stresses_py's own reader: the angles, the length, the width, the slip and
  the rectangle's centre (within a few metres in map view and 0.1 m in depth) must come back as given.

**Done when** `st.check("M54-T2", func=patch_line)` passes on 30 made-up faults (all strikes and rakes, dips 5-89.99,
lengths 0.5-30 km) placed anywhere in and around Korea.

**Stuck?** `st.hint("M54-T2", 1)`

In [ ]:
# Before you code (then read the reader): which point of the rectangle do you expect the longitude, latitude and
# depth of a Source_Patch line to describe (the centre, the hypocentre, a corner?), and in which order do you expect
# strike, dip and rake? For a fault 10 km long and 8 km wide dipping 60 degrees, how far (in map view and in depth)
# could the rectangle end up from where you meant it if you guessed the point wrong?
st.predict("M54-T2", """

""")

In [ ]:
def patch_line(strike_deg, dip_deg, rake_deg, length_km, width_km, center_lon, center_lat, center_depth_km, slip_m):
    """Elastic_stresses_py 'Source_Patch:' line for the rectangle whose centre is given.

    strike_deg, dip_deg, rake_deg : the fault (degrees)
    length_km, width_km           : length along strike and width down dip (km)
    center_lon, center_lat        : the centre of the rectangle (degrees)
    center_depth_km               : depth of the centre (km, positive down)
    slip_m                        : slip (m, positive)
    returns                       : one line of text that Elastic_stresses_py's reader turns into this fault
    """
    # Step 1: find in the reader the order of the fields and the point the line describes.
    # Step 2: the steps from the centre to that point: along strike, across strike in map view, and in depth.
    # Step 3: turn the east and north steps (km) into degrees near the centre; write the line.
    # YOUR CODE HERE
    raise NotImplementedError("M54-T2: write your code here")

In [ ]:
st.check("M54-T2", func=patch_line)

Programs place a rectangle by different points: its centre, the hypocentre, or a corner of the rectangle (glossary
`rupture_corner`). Your line for an invented fault, read back by Elastic_stresses_py (reference point = the centre
you gave, so its centre should come back near 0, 0); then the same numbers with the centre written where your line
has its point:

In [ ]:
line = patch_line(30.0, 60.0, 150.0, 10.0, 8.0, 127.5, 36.0, 10.0, 0.5)
print(line)
back = io_intxt.get_source_patch(line, 127.5, 36.0)
print("read back: centre", np.round(back.get_fault_center(), 4).tolist(), "km; top", round(back.top, 4),
      "km, bottom", round(back.bottom, 4), "km; slip", round(float(np.hypot(back.rtlat, back.reverse)), 6), "m")
fields = line.split()
naive = " ".join(fields[:6] + ["127.5", "36.0", "10.0"] + fields[9:])
wrong = io_intxt.get_source_patch(naive, 127.5, 36.0)
print("centre given as the line's point: read-back centre", np.round(wrong.get_fault_center(), 4).tolist(), "km")

**The same input error in two views.** Outlines and center dots are read back from the production program;
the star is the intended center. The right panel projects the rectangles onto east and depth, with depth
increasing downward. This is geometry, not a stress map. Keeping the same dimensions does not keep the same
location when a corner and a center are exchanged.

**Interpret:** estimate the horizontal and vertical displacement from the axes. At which receiver distances
would that placement error be a large fraction of the source-receiver separation?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.0, 4.0), constrained_layout=True)
for axis, depth_view in zip(axes, (False, True)):
    patch_projection(axis, back, "Your line, read back", "tab:blue", depth=depth_view)
    patch_projection(axis, wrong, "Center supplied as corner", "tab:orange", depth=depth_view)
    axis.plot(0.0, 10.0 if depth_view else 0.0, "*", color="k", ms=11, label="Intended center")
    axis.set(xlabel="East (km)", ylabel="Depth (km)" if depth_view else "North (km)", aspect="equal",
             title="East-depth projection" if depth_view else "Map projection")
    axis.grid(alpha=0.2)
axes[1].invert_yaxis()
axes[0].legend(fontsize=8)
show_figure(fig, "M54_reference_point_geometry.png")

**Explain (3-5 sentences):** Which point of the rectangle and which order of angles did you find in the reader,
and how did that compare with your prediction? Read the last printed line: how far did the rectangle move (in map
view and in depth) when the centre was written as the line's point, and how does that compare with the size of the
fault? Why can such a shift matter for a stress calculation at an aftershock 2 km from the fault but hardly at
one 50 km away?

## Part 3 · Stress around a rectangular fault: pyrocko's Okada code (about 60 min, then task 3)

Slip on a rectangle in a uniform elastic body with a flat free surface, an **elastic half-space** (glossary
`elastic_half_space`), has a closed-form solution: Okada (1985) for the surface and Okada (1992) for any point at
depth (glossary `okada_solution`). It gives the derivatives of the displacement; the **strain** (glossary `strain`)
ε is their symmetric part, ε = (∇u + ∇uᵀ)/2. **Hooke's law** (glossary `hookes_law`) turns strain into stress
with two **Lamé parameters** (glossary `lame_parameter`), λ and the **shear modulus** μ (glossary `shear_modulus`):

$$\sigma = \lambda\,\mathrm{tr}(\varepsilon)\,I + 2\mu\,\varepsilon .$$

With λ = μ, **Poisson's ratio** (glossary `poisson_ratio`) is 0.25, the usual choice for the crust.

Two programs compute this field, independently. Elastic_stresses_py splits each rectangle into two triangles and
uses cutde (Nikkhoo and Walter 2015). pyrocko (Heimann et al. 2017) wraps Okada's own equations in
`pyrocko.modelling.okada_ext.okada`. Its inputs are written for Okada's conventions, not for Elastic_stresses_py's:
read the docstrings of `OkadaSource.source_patch` and `OkadaSource.source_disloc` in `pyrocko.modelling.okada` (the
next cell prints where they are): a reference point in metres (north first), the edges `al1, al2` (along strike)
and `aw1, aw2` (along dip, up positive) measured from that point, and three dislocations. It returns twelve numbers
per point: three displacements and nine displacement derivatives, in north, east, down. It wants C-contiguous
arrays (`numpy.ascontiguousarray`).

In [ ]:
from pyrocko.modelling import okada as pyrocko_okada
from pyrocko.modelling import okada_ext
print("pyrocko's Okada source class:", short_path(pyrocko_okada.__file__))
print("okada_ext.okada(source_patches, dislocations, receivers, lame1, mu, nthreads=..., rotate_sdn=...)")

### M54-T3 · Stress around a rectangular fault with pyrocko's Okada code · core · about 60 min

**Goal:** write `okada_stress(fault, points_km, lame1_pa, mu_pa)`: the stress change at points around one slipping
rectangle, from pyrocko's `okada_ext.okada`, in the axes and units of Elastic_stresses_py.

**Given:** Part 3, pyrocko and numpy. Not allowed: Elastic_stresses_py, tectonic-utils, cutde and the course
package (the check compares with Elastic_stresses_py).

**Contract:**
- Inputs: `fault`, a dict with `strike`, `dip`, `rake` (degrees), `length_km`, `width_km`, `east_km`, `north_km`,
  `depth_km` (the **centre** of the rectangle in a local frame: km east and north of an origin, km below the
  surface) and `slip_m`; `points_km`, an array (n, 3) of east, north, depth (km, depth positive down, below the
  surface); `lame1_pa` (λ) and `mu_pa` (μ) in Pa.
- Output: a numpy array (n, 3, 3) of stress changes in Pa, rows and columns east, north, up, tension positive.
- Example (invented): a vertical fault striking north, rake 180, 2 km by 2 km, centre at (0, 0, 5 km), slip 0.1 m,
  λ = μ = 30 GPa, at the point (3, 1, 5) km gives, in kPa, [[87.2727, -2.3816, -0.2409], [-2.3816, -7.3839,
  -1.2889], [-0.2409, -1.2889, -10.8385]].

**Done when** `st.check("M54-T3", func=okada_stress)` passes on 16 made-up faults (lengths 0.5-20 km, all strikes
and rakes, dips 10-88, 12 points each away from the fault plane), compared point by point with Elastic_stresses_py's
stress (the cell after the check says how close it had to be).

**Stuck?** `st.hint("M54-T3", 1)`

In [ ]:
# Before you code: if you put pyrocko's reference point at the centre of the rectangle, what are al1, al2, aw1 and
# aw2 for a fault 4 km long and 2 km wide? For rake 180 and slip 1 m, what are the two shear dislocations pyrocko
# wants? And by how much do you expect pyrocko and Elastic_stresses_py to differ: 1e-1, 1e-3 or 1e-8 of the stress?
st.predict("M54-T3", """

""")

In [ ]:
def okada_stress(fault, points_km, lame1_pa, mu_pa):
    """Stress change (Pa; east, north, up; tension positive) at points around a slipping rectangle (pyrocko).

    fault     : dict with strike, dip, rake (deg), length_km, width_km, east_km, north_km, depth_km (centre), slip_m
    points_km : (n, 3) east, north, depth (km, depth positive down)
    lame1_pa, mu_pa : Lame's first parameter and shear modulus (Pa)
    returns   : (n, 3, 3) stress tensors in Pa
    """
    # Step 1: pyrocko's nine numbers of the rectangle (metres; north, east, depth of the reference point, strike,
    #         dip, al1, al2, aw1, aw2) and its three dislocations.
    # Step 2: the points as north, east, depth in metres (C-contiguous); okada_ext.okada with one thread.
    # Step 3: the derivatives (last nine numbers per point) as 3 x 3; the strain; turn it into east, north, up.
    # Step 4: Hooke's law (Part 3).
    # YOUR CODE HERE
    raise NotImplementedError("M54-T3: write your code here")

In [ ]:
st.check("M54-T3", func=okada_stress)

The check accepted a difference of at most 1e-5 of each made-up case's largest stress, at every point.

**R2: two programs, one fault.** An invented 10-km right-lateral fault (rake 180), nearly vertical (dip 89.9;
Elastic_stresses_py refuses exactly 90), written for Elastic_stresses_py with your `patch_line` and given to pyrocko
with your `okada_stress`, on a grid of points at 8 km depth. The largest difference is printed relative to the
largest stress on the grid. Then the classic map: ΔCFS on receivers parallel to the source (King et al. 1994,
figure 2), from your two functions.

In [ ]:
from elastic_stresses_py.PyCoulomb import configure_calc, run_dc3d
from elastic_stresses_py.PyCoulomb.disp_points_object.disp_points_object import Displacement_points
from elastic_stresses_py.PyCoulomb.inputs_object.input_obj import Input_object
import contextlib
import io
import warnings

TOY = {"strike": 0.0, "dip": 89.9, "rake": 180.0, "length_km": 10.0, "width_km": 8.0, "east_km": 0.0,
       "north_km": 0.0, "depth_km": 8.0, "slip_m": 1.0}
TOY_LON, TOY_LAT = 127.0, 36.0        # an invented place: the local frame's origin
gx, gy = np.meshgrid(np.linspace(-25.0, 25.0, 51), np.linspace(-25.0, 25.0, 51))
grid = np.column_stack([gx.ravel(), gy.ravel(), np.full(gx.size, 8.0)])
on_plane = (np.abs(grid[:, 0]) < 0.6) & (np.abs(grid[:, 1]) <= 5.6)
grid = grid[~on_plane]
esp_toy = io_intxt.get_source_patch(patch_line(TOY["strike"], TOY["dip"], TOY["rake"], TOY["length_km"],
                                               TOY["width_km"], TOY_LON, TOY_LAT, TOY["depth_km"], TOY["slip_m"]),
                                    TOY_LON, TOY_LAT)
toy_inputs = Input_object(PR1=0.25, FRIC=FRICTION, depth=0, start_gridx=-30, finish_gridx=30, start_gridy=-30,
                          finish_gridy=30, xinc=1, yinc=1, minlon=TOY_LON - 0.3, maxlon=TOY_LON + 0.3, zerolon=TOY_LON,
                          minlat=TOY_LAT - 0.3, maxlat=TOY_LAT + 0.3, zerolat=TOY_LAT, source_object=[esp_toy],
                          receiver_object=[])
with contextlib.redirect_stdout(io.StringIO()), warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)     # cutde's note that it copies an array
    toy_params = configure_calc.Params(mu=SHEAR_MODULUS_PA, lame1=LAME1_PA, plot_grd_disp=False)
    esp_strain = np.array(run_dc3d.compute_xy_strain(
        toy_inputs, toy_params, [Displacement_points(lon=e, lat=n, depth=z) for e, n, z in grid]))
esp_sigma = esp_math.stress_tensor_batch(esp_strain, LAME1_PA, SHEAR_MODULUS_PA)
mine_sigma = okada_stress(TOY, grid, LAME1_PA, SHEAR_MODULUS_PA)
print(f"{len(grid)} points; largest |difference| / largest |stress| = "
      f"{np.abs(mine_sigma - esp_sigma).max() / np.abs(esp_sigma).max():.2e}")

**One stress tensor, three spatial contributions.** Black marks the source's map projection and dotted
lines mark zero. Receivers are parallel to that source at 8 km depth. The idealized fault cells are masked;
color saturation near it should not be read as a precise resolved stress. The same symmetric scale makes
shear and effective-normal contributions comparable before you inspect their sum.

**Interpret:** where do shear and normal terms reinforce each other, and where do they cancel? Why can
changing receiver orientation or friction move the positive and negative regions without changing source slip?

In [ ]:
full = np.column_stack([gx.ravel(), gy.ravel(), np.full(gx.size, 8.0)])
_, _, cfs_toy = coulomb_change(okada_stress(TOY, full, LAME1_PA, SHEAR_MODULUS_PA), TOY["strike"], TOY["dip"],
                               TOY["rake"], FRICTION, SKEMPTON)
# All panels use the same receiver and symmetric color limits. Mask cells on the idealized fault.
toy_tau, toy_normal, _ = coulomb_change(okada_stress(TOY, full, LAME1_PA, SHEAR_MODULUS_PA),
                                      TOY["strike"], TOY["dip"], TOY["rake"], FRICTION, SKEMPTON)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.9), constrained_layout=True)
for ax, values, title in zip(axes, (toy_tau, FRICTION * toy_normal, cfs_toy),
                              ("Shear contribution", "Friction × effective normal", "Coulomb change: their sum")):
    field = np.where(on_plane.reshape(gx.shape), np.nan, np.asarray(values).reshape(gx.shape))
    mesh = ax.pcolormesh(gx, gy, field, cmap="RdBu_r", vmin=-100, vmax=100, shading="auto")
    zero_boundary(ax, gx, gy, field)
    ax.plot([0, 0], [-5, 5], color="k", lw=2)
    ax.set(xlabel="East (km)", ylabel="North (km)", title=title, aspect="equal")
fig.colorbar(mesh, ax=axes, label="Stress change (kPa); colors saturate at ±100", extend="both", shrink=0.85)
show_figure(fig, "M54_toy_stress_components.png")
print(f"at 8 km depth: largest {cfs_toy.max():.1f} kPa, smallest {cfs_toy.min():.1f} kPa (on the grid, the fault "
      "cells included)")

**Friction at a few points.** Your two functions again, at five points of the same map (8 km deep, receivers
parallel to the source): 3 km beyond the north end of the fault on its line; 1 km beyond that end and 4 km to the
east and to the west of the line; and 4 km to each side of the fault's middle. The table gives the shear, the
normal stress and the Coulomb change for three frictions.

In [ ]:
probe = np.array([[0.0, 8.0, 8.0], [4.0, 6.0, 8.0], [-4.0, 6.0, 8.0], [4.0, 0.0, 8.0], [-4.0, 0.0, 8.0]])
probe_sigma = okada_stress(TOY, probe, LAME1_PA, SHEAR_MODULUS_PA)
probe_rows = []
for probe_friction in (0.0, FRICTION, 0.8):
    p_shear, p_normal, p_cfs = coulomb_change(probe_sigma, TOY["strike"], TOY["dip"], TOY["rake"], probe_friction,
                                              SKEMPTON)
    for (p_east, p_north, _), v_s, v_n, v_c in zip(probe, p_shear, p_normal, p_cfs):
        probe_rows.append({"friction": probe_friction, "east_km": p_east, "north_km": p_north, "shear_kpa": v_s,
                           "normal_kpa": v_n, "coulomb_kpa": v_c})
print(pd.DataFrame(probe_rows).round(2).to_string(index=False))

**Friction changes the balance, not the source stress tensor.** Each line follows one of the five map
locations above. The intercept is its shear contribution; the slope reflects its effective normal stress.
Here the source and receiver geometry, slip and pore-pressure parameter stay fixed.

**Interpret:** which lines cross zero over this friction range? Which locations would retain their sign,
and which would require a better constraint on friction before a triggering claim?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8), constrained_layout=True)
visual_probe = pd.DataFrame(probe_rows)
for (east, north), part in visual_probe.groupby(["east_km", "north_km"], sort=False):
    ax.plot(part["friction"], part["coulomb_kpa"], "o-", label=f"E {east:g}, N {north:g} km")
ax.axhline(0, color="0.3", lw=0.8)
ax.set(xlabel="Friction coefficient", ylabel="Coulomb stress change (kPa)", title="Your stress resolution at fixed locations")
ax.legend(fontsize=8, ncol=2)
show_figure(fig, "M54_friction_sensitivity.png")

**Explorer E10 (optional).** Move the receiver's strike, dip and rake, the friction, and the source's rake and
slip, and watch the lobes of shear, normal and Coulomb change in map view. The stress comes from your
`okada_stress` and the resolving from your `coulomb_change`; the explorer computes neither itself. Nothing below
depends on it.

In [ ]:
try:
    explorer = st.widgets.coulomb_explorer(resolve=coulomb_change, stress=okada_stress, source=TOY,
                                           receiver=(0.0, 90.0, 180.0), friction=FRICTION, lame1_pa=LAME1_PA,
                                           mu_pa=SHEAR_MODULUS_PA)
    display(explorer)
except Exception as exc:  # noqa: BLE001 - optional cell: the module does not depend on it
    print("The explorer is not available here:", exc)

**Explain (3-5 sentences):** Read your printed difference between the two programs: is it at the level you
predicted, and what does it say about the conventions you chose in both functions (reference point, slip sense,
axes)? In your printed table: at which points is the Coulomb change positive and at which negative, and does your
map agree? Which of the three columns change with friction and which do not? At which points does the friction
change the size or the sign of the Coulomb change, and is the normal stress or the shear larger there?

## Part 4 · How big is the fault? Moment, stress drop and scaling (about 40 min, then task 4)

A stress calculation needs a fault size and a slip, but a catalog gives a magnitude. The **seismic moment**
(glossary `seismic_moment`) is M0 = μ A D (shear modulus × area × mean slip); your M52 function turns a **moment
magnitude** (glossary `moment_magnitude`) into M0. One more number links area and slip: the **stress drop**
(glossary `stress_drop`) Δσ, the drop of shear stress on the fault during the earthquake. For a **circular crack**
(glossary `circular_crack`) of radius a, Eshelby (1957) gives

$$M_0 = \frac{16}{7}\,\Delta\sigma\,a^3 .$$

Measured stress drops of crustal earthquakes spread over about 0.1-100 MPa, with typical values of a few MPa
(Shearer 2019). Elastic_stresses_py and pyrocko want rectangles; we use the square of the same area.

A second route is an empirical **scaling relation** (glossary `scaling_relation`): Wells and Coppersmith (1994)
fitted the subsurface rupture length and the down-dip width of many earthquakes against magnitude, with a scatter of
about 0.15 in log10; Leonard (2010) gave relations that keep length, width, slip and moment consistent, with their
own constants for stable continental regions such as Korea. Part 6 compares both routes for Gyeongju.

### M54-T4 · Fault size and slip from moment and stress drop · core · about 30 min

**Goal:** write `square_fault(m0_nm, stress_drop_pa, shear_modulus_pa)`: the side of the square with the area of
Eshelby's circular crack, and the mean slip.

**Given:** Part 4 and numpy. Not allowed: Elastic_stresses_py, tectonic-utils, pyrocko and the course package.

**Contract:**
- Inputs: seismic moment (N m), stress drop (Pa), shear modulus (Pa); numbers or numpy arrays of one shape.
- Output: a tuple `(side_km, slip_m)`: floats for numbers, arrays for arrays.
- Example (invented): `square_fault(2.5e20, 6e6, 3.2e10)` gives (46.648, 3.5903).

**Done when** `st.check("M54-T4", func=square_fault)` passes on 200 made-up cracks (radii 30 m to 30 km, stress
drops 0.1-100 MPa), numbers and arrays.

**Stuck?** `st.hint("M54-T4", 1)`

In [ ]:
# Before you code: about how long is the side of the fault of the Mw 3.8 Sangju earthquake for a stress drop of 3 MPa:
# 100 m, 1 km or 10 km? How large is its slip? Then: at an aftershock 5 km away, do you expect its Coulomb stress
# change to be larger or smaller than the tidal stresses of about 1 kPa that the solid Earth feels twice a day?
st.predict("M54-T4", """

""")

In [ ]:
def square_fault(m0_nm, stress_drop_pa, shear_modulus_pa):
    """Side (km) of the square with the area of Eshelby's circular crack, and its mean slip (m).

    m0_nm            : seismic moment (N m)
    stress_drop_pa   : stress drop (Pa)
    shear_modulus_pa : shear modulus (Pa)
    returns          : (side_km, slip_m), floats for numbers, arrays for arrays
    """
    # Step 1: the crack's radius from Part 4's formula (m).
    # Step 2: the square of the same area; its side in km.
    # Step 3: the slip from M0 = mu A D.
    # YOUR CODE HERE
    raise NotImplementedError("M54-T4: write your code here")

In [ ]:
st.check("M54-T4", func=square_fault)

**R2 on real data: Sangju 2019.** The KIGAM mechanism (plane A; which plane slipped is not known) and Mw of the
ML 3.9 mainshock, its size from your `square_fault`, and the Coulomb change on receivers like the mainshock at the
later earthquakes of the KMA catalog within 10 km (UTC), from your `okada_stress` and `coulomb_change`. The local
frame puts the mainshock's KMA epicentre at (0, 0); its depth is KIGAM's.

In [ ]:
from tectonic_utils.geodesy import fault_vector_functions as esp_geo


def to_local(lon, lat, lon0, lat0):
    """km east and north of (lon0, lat0), with Elastic_stresses_py's own rule, so that both programs see the same
    points."""
    east, north = esp_geo.latlon2xy(list(np.atleast_1d(lon)), list(np.atleast_1d(lat)), lon0, lat0)
    return np.asarray(east, dtype=float), np.asarray(north, dtype=float)


sangju = st.data.sequence("sangju2019")
sj_main = sangju.loc[sangju["event_id"] == SANGJU_EVENT_ID].iloc[0]
sj_after = sangju[sangju["origin_utc"] > sj_main["origin_utc"]].reset_index(drop=True)
sj_m0 = magnitude_moment(mw=SANGJU_MW)
sj_side, sj_slip = square_fault(sj_m0, STRESS_DROP_MPA * 1e6, SHEAR_MODULUS_PA)
print(f"Sangju: Mw {SANGJU_MW} -> M0 {sj_m0:.3g} N m; stress drop {STRESS_DROP_MPA} MPa -> square side "
      f"{sj_side:.3f} km, slip {100 * sj_slip:.2f} cm")
SJ_FAULT = {"strike": SANGJU_PLANE[0], "dip": SANGJU_PLANE[1], "rake": SANGJU_PLANE[2], "length_km": sj_side,
            "width_km": sj_side, "east_km": 0.0, "north_km": 0.0, "depth_km": SANGJU_DEPTH_KM, "slip_m": sj_slip}
sj_e, sj_n = to_local(sj_after["longitude"].to_numpy(), sj_after["latitude"].to_numpy(),
                      float(sj_main["longitude"]), float(sj_main["latitude"]))
sj_after["east_km"], sj_after["north_km"] = sj_e, sj_n
known = sj_after["depth_km"].notna()
pts_sj = sj_after.loc[known, ["east_km", "north_km", "depth_km"]].to_numpy(float)
_, _, sj_cfs = coulomb_change(okada_stress(SJ_FAULT, pts_sj, LAME1_PA, SHEAR_MODULUS_PA), *SANGJU_PLANE, FRICTION,
                              SKEMPTON)
sj_table = sj_after.loc[known, ["origin_utc", "ml", "depth_km"]].copy()
sj_table["distance_km"] = np.sqrt(pts_sj[:, 0] ** 2 + pts_sj[:, 1] ** 2 + (pts_sj[:, 2] - SANGJU_DEPTH_KM) ** 2)
sj_table["dcfs_kpa"] = sj_cfs
sj_table["above_tides"] = np.abs(sj_cfs) > TIDAL_KPA
print(f"{len(sj_after)} later earthquakes in the subset, {int(known.sum())} with a depth; "
      f"{int((~known).sum())} without one are left out")
print(sj_table.sort_values("distance_km").round(3).to_string(index=False))

In [ ]:
print("median and largest |Coulomb change| on rings around the Sangju source at its depth (72 directions):")
for radius in (1.0, 2.0, 3.0, 5.0, 10.0):
    az = np.radians(np.arange(0.0, 360.0, 5.0))
    ring = np.column_stack([radius * np.sin(az), radius * np.cos(az), np.full(az.size, SANGJU_DEPTH_KM)])
    _, _, c_ring = coulomb_change(okada_stress(SJ_FAULT, ring, LAME1_PA, SHEAR_MODULUS_PA), *SANGJU_PLANE, FRICTION)
    print(f"  {radius:4.1f} km: median {np.median(np.abs(c_ring)):8.3f} kPa, largest {np.abs(c_ring).max():8.3f} kPa"
          f"  (tides about {TIDAL_KPA} kPa)")
lat_step = np.diff(np.unique(np.round(sangju["latitude"].to_numpy(), 6))).min()
print(f"KMA coordinates in this subset step by {lat_step:.3f} degrees ({lat_step * 111.2:.2f} km in latitude); "
      f"depths are whole kilometres: {bool(np.all(sangju['depth_km'].dropna() % 1 == 0))}")



**Stress has a spatial scale.** At each radius the shaded band spans the 5th–95th percentiles of the absolute
modeled Coulomb change over 72 directions at the source depth. The median can conceal large directional
differences. Dots are the later catalog events evaluated at their own depths; they are not measured stresses.
The tidal reference is an order-of-magnitude comparison, not a universal triggering threshold.

**Interpret:** at what distances are location uncertainty and receiver orientation likely to matter most?
Why does placing an aftershock on a modeled positive lobe not demonstrate that the source triggered it?

In [ ]:
visual_radii = np.geomspace(max(0.5, sj_side), 25, 24)
visual_ring = []
visual_az = np.radians(np.arange(0, 360, 5))
for radius in visual_radii:
    visual_points = np.column_stack([radius * np.sin(visual_az), radius * np.cos(visual_az),
                                      np.full(visual_az.size, SANGJU_DEPTH_KM)])
    visual_cfs = coulomb_change(okada_stress(SJ_FAULT, visual_points, LAME1_PA, SHEAR_MODULUS_PA),
                                 *SANGJU_PLANE, FRICTION, SKEMPTON)[2]
    visual_ring.append(np.percentile(np.abs(visual_cfs), [5, 50, 95]))
visual_ring = np.asarray(visual_ring)
fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
ax.fill_between(visual_radii, visual_ring[:, 0], visual_ring[:, 2], alpha=0.2, label="5–95% across directions")
ax.loglog(visual_radii, visual_ring[:, 1], label="Directional median")
ax.scatter(sj_table["distance_km"], np.maximum(np.abs(sj_table["dcfs_kpa"]), 1e-12), color="k", s=25,
           label="Later catalog events, own depths")
ax.axhline(TIDAL_KPA, color="tab:red", ls="--", label="Illustrative tidal scale")
ax.set(xscale="log", yscale="log", xlabel="Distance from source center (km)", ylabel="|Coulomb change| (kPa)",
       title="Sangju 2019 teaching example: modeled stress and distance")
ax.legend(fontsize=8)
show_figure(fig, "M54_stress_distance.png")

Tidal stresses: the solid-Earth tides strain the crust by a few parts in 10⁸ twice a day (Agnew 2015), which times a
shear modulus of 30 GPa is about 1 kPa (glossary `tidal_stress`); ocean loading adds more near coasts. Tidal stresses
of this size correlate with earthquakes only weakly and only in special cases (Vidale et al. 1998; Cochran et al.
2004). The location of a KMA epicentre is known to about a kilometre or worse (glossary `location_uncertainty`):
the last printed line says how finely this catalog writes it.

**Explain (3-5 sentences):** Read your printed side and slip for Sangju and compare them with your prediction.
Using the printed table and rings, at which distances is the Coulomb change of the Sangju mainshock larger than the
tides, and how many of the later earthquakes lie there? Compare the fault size with the catalog's step in latitude
and depth: could you tell on which side of a lobe the near events lie? Say plainly what these numbers allow you to
conclude about stress triggering at Sangju, and what they do not.

## Part 5 · Is the fraction of aftershocks in positive areas special? A rotation test (about 40 min, then task 5)

A Coulomb map is often judged by the fraction of aftershocks that fall where ΔCFS > 0. A fraction alone says little:
it must be compared with the fraction that aftershocks would reach if the map had nothing to do with them. A
**null hypothesis test** (glossary `null_hypothesis_test`) builds such a "nothing" by hand. Here it is a **rotation
null** (glossary `rotation_null`): turn the whole aftershock cloud about the mainshock by a random angle (keeping
every distance and depth), evaluate the map at the turned points, and repeat many times. The turned clouds show
which fractions occur by chance for a cloud of that shape at those distances. A natural first guess for this chance
level is 50 %; the test measures it. The **p-value** (glossary `p_value`) is the share of turns that did at least as
well as the real cloud, with one added above and below so that it is never zero:
p = (1 + number of turns with fraction ≥ observed) / (number of turns + 1).

Conventions of this module: a turn is clockwise seen from above, so each point's compass direction from the centre
grows by the angle; points where the map is NaN (too close to a fault to trust) are left out of every fraction,
before and after each turn.

### M54-T5 · A rotation test for aftershocks in positive areas · core · about 45 min

**Goal:** write `rotation_test(dcfs_at, east_km, north_km, depth_km, center_east_km, center_north_km, n_rotations,
rng)`.

**Given:** Part 5 and numpy. Not allowed: the course package.

**Contract:**
- Inputs: `dcfs_at(east_km, north_km, depth_km)`, a function that returns the Coulomb change (kPa) at arrays of
  points, NaN where a point is to be left out; the points' coordinates (arrays of length n, km); the centre of the
  turns (km); `n_rotations`; `rng`, a numpy Generator.
- Draw all angles first, with one call `rng.uniform(0.0, 360.0, n_rotations)` (degrees). Call `dcfs_at` once for
  the points as they are and once per turn, each time with arrays of length n and the depths unchanged.
- Output: a dict with `observed` (the fraction of finite values above zero at the points as they are), `null` (array
  of `n_rotations` fractions, one per turn, in the order of the angles), `angles_deg` (the angles) and `p_value`
  (Part 5).

**Done when** `st.check("M54-T5", func=rotation_test)` passes on 12 made-up maps whose positive fraction after any
turn is known by construction.

**Stuck?** `st.hint("M54-T5", 1)`

In [ ]:
# Before you code: think of the toy map of Part 3 and points scattered at random around the fault. Would the chance
# level of the rotation test be 50 %? Name two things that could move it away from 50 % (the map, the receivers, the
# friction, the shape of the cloud, the points left out).
st.predict("M54-T5", """

""")

In [ ]:
def rotation_test(dcfs_at, east_km, north_km, depth_km, center_east_km, center_north_km, n_rotations, rng):
    """Fraction of points with positive Coulomb change, against the fractions of the cloud turned about a centre.

    dcfs_at   : function (east_km, north_km, depth_km) -> Coulomb change (kPa), NaN where a point is left out
    east_km, north_km, depth_km : the points (arrays of length n)
    center_east_km, center_north_km : centre of the turns (km)
    n_rotations : number of turns; rng : numpy Generator
    returns   : dict with observed, null, angles_deg, p_value
    """
    # Step 1: the angles, all at once, from rng.
    # Step 2: the fraction at the points as they are (finite values only).
    # Step 3: for each angle, turn the points clockwise about the centre and keep the fraction.
    # Step 4: the p-value of Part 5.
    # YOUR CODE HERE
    raise NotImplementedError("M54-T5: write your code here")

In [ ]:
st.check("M54-T5", func=rotation_test)

Your test on the toy map of Part 3, with 300 invented points scattered around the fault (and left out within 2 km
of it), twice: points spread evenly in direction, and points stretched along the fault.

In [ ]:
def toy_dcfs(east_km, north_km, depth_km):
    pts = np.column_stack([east_km, north_km, depth_km])
    _, _, c = coulomb_change(okada_stress(TOY, pts, LAME1_PA, SHEAR_MODULUS_PA), TOY["strike"], TOY["dip"],
                             TOY["rake"], FRICTION, SKEMPTON)
    near = (np.abs(pts[:, 0]) < 2.0) & (np.abs(pts[:, 1]) < 7.0)
    return np.where(near, np.nan, c)


demo_rng = np.random.default_rng(7)
toy_rotation_examples = []
for label, stretch in (("even in direction", 1.0), ("stretched along the fault", 3.0)):
    r = demo_rng.uniform(3.0, 20.0, 300)
    a = demo_rng.uniform(0.0, 2.0 * np.pi, 300)
    de, dn = r * np.sin(a), stretch * r * np.cos(a)
    res = rotation_test(toy_dcfs, de, dn, np.full(300, 8.0), 0.0, 0.0, 200, np.random.default_rng(1))
    toy_rotation_examples.append((label, res))
    print(f"{label:27s}: observed {res['observed']:.3f}, null mean {np.mean(res['null']):.3f} "
          f"(5-95 %: {np.percentile(res['null'], 5):.3f}-{np.percentile(res['null'], 95):.3f}), p {res['p_value']:.3f}")



**The null distribution depends on the spatial sample.** Each histogram comes from rotating the same cloud
of points, not drawing new independent earthquakes. The solid line is the observed positive fraction;
the dashed line is 0.5, shown only as a reference. Your test compares the observation with the actual
rotation distribution, whose chance level need not be one half.

**Interpret:** why can stretching the cloud change the null distribution? Which spatial characteristics
does the rotation preserve, and which alternatives to stress triggering does this test leave open?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.5), constrained_layout=True)
for ax, (label, result) in zip(axes, toy_rotation_examples):
    ax.hist(result["null"], bins=np.linspace(0, 1, 26), color="0.65", edgecolor="white")
    ax.axvline(result["observed"], color="tab:red", lw=2, label="Observed")
    ax.axvline(0.5, color="0.2", ls="--", lw=0.8, label="One half")
    ax.set(xlabel="Fraction in positive modeled stress", ylabel="Number of rotations", title=label)
    ax.legend(fontsize=8)
show_figure(fig, "M54_toy_rotation_null.png")

**Explain (3-5 sentences):** Read your two printed rows: what chance level (null mean) did the toy map give for each
cloud, and is it 50 %? Which of the causes you predicted do the two rows support? Why must the chance level be
measured for the real cloud and the real map rather than assumed, and what would a test that assumed 50 % conclude
wrongly?

## Part 6 · Gyeongju 2016: two faults, Elastic_stresses_py, and the aftershocks (about 120 min, then task 6)

The ML 5.1 foreshock and the ML 5.8 mainshock of 12 September 2016 were the largest instrumentally recorded
earthquakes on land in Korea at the time (Hong et al. 2017). Aftershocks show two closely spaced, parallel faults
striking north-north-east, oblique to the Yangsan fault (Uchide and Song 2018; Woo et al. 2019). Uchide and Song
(2018) found that the mainshock ruptured toward the north-north-east and the foreshock toward the south-south-west,
with stress drops up to 62 and 43 MPa on parts of the faults. Here you build both as uniform rectangles, the
simplest model a Coulomb study can use, and you test it against the KMA aftershocks.

**Mechanisms** come from moment tensors: the USGS (ComCat, through ObsPy's FDSN client; glossary `fdsn`) for the
mainshock. The next cell asks the USGS for the foreshock too and prints what it finds; the Global CMT solution
(Dziewonski et al. 1981; Ekström et al. 2012) of the foreshock comes through the ISC Bulletin's FDSN service. The
answers are kept in `results/gyeongju_mechanisms.json`, so a rerun needs no network.

In [ ]:
mech_file = Path(RESULT_DIR) / "gyeongju_mechanisms.json"
if mech_file.is_file():
    MECH = json.loads(mech_file.read_text(encoding="utf-8"))
    print(f"read {mech_file} (fetched {MECH['fetched_utc']})")
else:
    from obspy import UTCDateTime
    from obspy.clients.fdsn import Client

    def planes_of(fm):
        nps = fm.nodal_planes
        return [[float(p.strike), float(p.dip), float(p.rake)] for p in (nps.nodal_plane_1, nps.nodal_plane_2)]

    usgs = Client("USGS", timeout=60)
    main_ev = usgs.get_events(eventid=USGS_MAIN_EVENT)[0]
    main_fm = main_ev.preferred_focal_mechanism() or main_ev.focal_mechanisms[0]
    fore_usgs = usgs.get_events(eventid=USGS_FORE_EVENT)[0]
    fore_usgs_mt = [fm for fm in fore_usgs.focal_mechanisms if fm.moment_tensor is not None and
                    fm.moment_tensor.scalar_moment]
    isc = Client("ISC", timeout=60)
    fore_isc = isc.get_events(starttime=UTCDateTime(FORE_WINDOW_UTC[0]), endtime=UTCDateTime(FORE_WINDOW_UTC[1]),
                              minlatitude=35.0, maxlatitude=36.5, minlongitude=128.5, maxlongitude=130.0,
                              includeallorigins=True)[0]
    gcmt = [fm for fm in fore_isc.focal_mechanisms if fm.creation_info and fm.creation_info.author == "GCMT"
            and fm.nodal_planes is not None and fm.moment_tensor is not None]
    MECH = {"fetched_utc": str(UTCDateTime().strftime("%Y-%m-%dT%H:%M:%SZ")),
            "main": {"source": f"USGS {main_fm.resource_id.id.split('/')[-3]}", "planes": planes_of(main_fm),
                     "m0_nm": float(main_fm.moment_tensor.scalar_moment)},
            "fore": {"source": "GCMT via the ISC Bulletin", "planes": planes_of(gcmt[0]),
                     "m0_nm": float(gcmt[0].moment_tensor.scalar_moment)},
            "fore_usgs_moment_tensors": len(fore_usgs_mt)}
    mech_file.write_text(json.dumps(MECH, indent=1), encoding="utf-8")
for key in ("fore", "main"):
    m = MECH[key]
    print(f"{key}: {m['source']}: planes {m['planes']}, M0 {m['m0_nm']:.4g} N m = Mw "
          f"{magnitude_moment(m0_nm=m['m0_nm']):.2f}")
print("moment tensors that the USGS lists for the foreshock:", MECH["fore_usgs_moment_tensors"])

In [ ]:
gyeongju = st.data.sequence("gyeongju2016")
g_main = gyeongju.loc[gyeongju["event_id"] == GYEONGJU_MAIN_ID].iloc[0]
g_fore = gyeongju.loc[gyeongju["event_id"] == GYEONGJU_FORE_ID].iloc[0]
LON0, LAT0 = float(g_main["longitude"]), float(g_main["latitude"])
after = gyeongju[(gyeongju["origin_utc"] > g_main["origin_utc"]) &
                 (gyeongju["origin_utc"] <= g_main["origin_utc"] + pd.Timedelta(days=AFTERSHOCK_DAYS)) &
                 (gyeongju["depth_km"] > 0)].reset_index(drop=True)
after["east_km"], after["north_km"] = to_local(after["longitude"].to_numpy(), after["latitude"].to_numpy(), LON0, LAT0)
print(f"KMA: foreshock {g_fore['origin_utc']} ML {g_fore['ml']}, mainshock {g_main['origin_utc']} ML {g_main['ml']}; "
      f"{len(after)} later events within {AFTERSHOCK_DAYS} days with a depth above 0 km")
fig, ax = plt.subplots(figsize=(4.8, 4.8))
ax.scatter(after["east_km"], after["north_km"], s=4, color="0.4", alpha=0.5, label="KMA aftershocks")
ax.plot(0.0, 0.0, marker="*", ms=14, color="crimson", ls="none", label="mainshock (KMA)")
ax.set(xlabel="East (km)", ylabel="North (km)", aspect="equal", title="Gyeongju 2016 aftershocks")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.08), ncol=2, frameon=False)
fig.savefig(Path(FIGURE_DIR) / "M54_gyeongju_aftershocks.png", dpi=150)
plt.show()

**Faults.** Keep the nodal plane that strikes like the aftershock cloud (`STRIKE_RANGE`). Size and slip come from
your `square_fault` at `STRESS_DROP_MPA`; the next cell prints them next to the Wells and Coppersmith (1994)
strike-slip relations (subsurface rupture length and down-dip width, with their ±1 standard deviation in log10,
0.15 and 0.14; computed with the copy in tectonic-utils). Each rectangle is centred at the KMA hypocentre of its
event: a simplification, since a hypocentre is where a rupture starts, not its middle.

In [ ]:
from tectonic_utils.seismo import wells_and_coppersmith as wc

FAULTS = {}
for key, row in (("fore", g_fore), ("main", g_main)):
    planes = MECH[key]["planes"]
    plane = next(p for p in planes if STRIKE_RANGE[0] <= p[0] % 360.0 <= STRIKE_RANGE[1])
    side, slip = square_fault(MECH[key]["m0_nm"], STRESS_DROP_MPA * 1e6, SHEAR_MODULUS_PA)
    east, north = to_local(float(row["longitude"]), float(row["latitude"]), LON0, LAT0)
    FAULTS[key] = {"strike": plane[0], "dip": plane[1], "rake": plane[2], "length_km": side, "width_km": side,
                   "east_km": float(east[0]), "north_km": float(north[0]), "depth_km": float(row["depth_km"]),
                   "slip_m": slip, "lon": float(row["longitude"]), "lat": float(row["latitude"])}
    mw = magnitude_moment(m0_nm=MECH[key]["m0_nm"])
    rld, rw = wc.RLD_from_M(mw, "SS"), wc.RW_from_M(mw, "SS")
    print(f"{key}: plane {plane}; your square: side {side:.2f} km, slip {slip:.3f} m; Wells & Coppersmith (Mw "
          f"{mw:.2f}): length {rld:.2f} km ({rld / 10 ** 0.15:.2f}-{rld * 10 ** 0.15:.2f}), width {rw:.2f} km "
          f"({rw / 10 ** 0.14:.2f}-{rw * 10 ** 0.14:.2f})")
RECEIVER = (FAULTS["main"]["strike"], FAULTS["main"]["dip"], FAULTS["main"]["rake"])
AUX = next(p for p in MECH["main"]["planes"] if tuple(p) != RECEIVER)
print("receiver faults like the mainshock plane:", RECEIVER, "; its auxiliary plane:", tuple(AUX))

**The R3 runs.** The cells after your task write Elastic_stresses_py's input file (a `General:` line with Poisson's
ratio, the friction and the map's limits and origin, your two `patch_line` lines, and a map of the Coulomb change
on receivers like the mainshock at `PROFILE_DEPTH_KM`) and a configuration file, put your aftershock points in a
points file (the program writes the full stress tensor at each, in `ll_stresses.txt`), and run the program three
times: `first`, `rerun` (the same files again, unchanged) and `changed` (only the friction of the `General:` line
set to `CHANGED_FRICTION`). Each run writes a folder; `horizontal_profile_stresses.txt` holds its map. A **checksum**
(glossary `checksum`; SHA-256 here) of each file shows whether two runs wrote the same bytes.

### M54-T6 · Gyeongju 2016, Elastic_stresses_py runs and the aftershocks · core · about 90 min

**Goal:** write two functions, `read_stresses(path)` and `dcfs_at(east_km, north_km, depth_km)`; the cells after
them run Elastic_stresses_py three times, compare it with your pyrocko route, and build your answer `gyeongju_report`.

**Given:** Parts 1-6, your functions of tasks 1-5, and `FAULTS`, `RECEIVER`, `FRICTION`, `EXCLUDE_KM`.

**Contract:**
- `read_stresses(path)`: reads a stress file of Elastic_stresses_py (`ll_stresses.txt`; its first line says what
  each column holds) and returns `(lon, lat, depth_km, stress)`: three arrays of length n and an array (n, 3, 3) of
  symmetric tensors in the unit that your `coulomb_change` takes, in the file's order.
- `dcfs_at(east_km, north_km, depth_km)`: the Coulomb change (kPa) at arrays of points of the local frame, from the
  stress of both faults in `FAULTS` (your `okada_stress`, summed), resolved with your `coulomb_change` on `RECEIVER`
  with `FRICTION` and `SKEMPTON`; NaN at points closer than `EXCLUDE_KM` to either rectangle (the distance to the
  nearest point of the rectangle, not of its infinite plane).
- The cells after them fill the rest of the answer (runs, fingerprints, points, sensitivity rows, the rotation
  test); run them unchanged.

**Done when** `st.check("M54-T6", answer=gyeongju_report)` passes: your `read_stresses` on made-up files written by
Elastic_stresses_py, your T1, T3 and T5 functions again on made-up cases, then your runs: fingerprints recomputed,
the rerun identical, the changed run differing in friction only, your `dcfs_at` equal to your own `okada_stress`
and `coulomb_change` (at your points and at made-up points around them) and within 1e-3 of the Coulomb change from
your first run's stresses, and your sensitivity rows and rotation test recomputed with your own functions.

**Stuck?** `st.hint("M54-T6", 1)`

In [ ]:
# Before you code and run: will the unchanged rerun write identical files? When only the friction changes, which of
# the two files (the stress tensors at the points, the map of the Coulomb change) will change, and why? Will the
# aftershocks beyond EXCLUDE_KM fall in positive areas more often than the turned clouds of the rotation test, and
# what chance level do you expect?
st.predict("M54-T6", """

""")

In [ ]:
def read_stresses(path):
    """(lon, lat, depth_km, stress) from Elastic_stresses_py's ll_stresses.txt; stress (n, 3, 3) in the unit that
    coulomb_change takes."""
    # Step 1: the numbers of the file (its first line is a comment); one row per point.
    # Step 2: the six stress values of a row into a symmetric 3 x 3 tensor (the first line gives their order),
    #         in the unit your coulomb_change takes.
    # YOUR CODE HERE
    raise NotImplementedError("M54-T6: write your code here")


def dcfs_at(east_km, north_km, depth_km):
    """Coulomb change (kPa) of both Gyeongju faults on RECEIVER at points of the local frame; NaN near a fault."""
    # Step 1: the points as an (n, 3) array; the stress of both faults from your okada_stress, summed.
    # Step 2: resolve it with your coulomb_change on RECEIVER with FRICTION and SKEMPTON.
    # Step 3: for each fault, the distance from each point to the nearest point of its rectangle (coordinates along
    #         strike and along dip from the centre, clipped to half the length and half the width, plus the
    #         distance off the plane); NaN where it is below EXCLUDE_KM.
    # YOUR CODE HERE
    raise NotImplementedError("M54-T6: write your code here")

In [ ]:
# The R3 runs: Elastic_stresses_py's files, three runs, the points and the answer. Run this cell unchanged.
ESP_DIR = Path(RESULT_DIR) / "esp_gyeongju"
ESP_DIR.mkdir(parents=True, exist_ok=True)
keep = np.isfinite(dcfs_at(after["east_km"].to_numpy(), after["north_km"].to_numpy(), after["depth_km"].to_numpy()))
pts_g = after.loc[keep, ["longitude", "latitude", "depth_km", "east_km", "north_km"]].reset_index(drop=True)
pts_g.columns = ["lon", "lat", "depth_km", "east_km", "north_km"]
points_csv = ESP_DIR / "points.csv"
pts_g.to_csv(points_csv, index=False)
(ESP_DIR / "points.txt").write_text("".join(f"{a:.6f} {b:.6f} {c:.6f}\n" for a, b, c in
                                            pts_g[["lon", "lat", "depth_km"]].to_numpy()), encoding="utf-8")


def esp_input(friction):
    lines = ["# Gyeongju 2016: foreshock and mainshock as uniform rectangles (module M54)",
             f"General: 0.25 {friction:.3f} {LON0 - 0.3:.4f} {LON0 + 0.3:.4f} {LON0:.4f} {LAT0 - 0.25:.4f} "
             f"{LAT0 + 0.25:.4f} {LAT0:.4f}"]
    for key in ("fore", "main"):
        f = FAULTS[key]
        lines.append(patch_line(f["strike"], f["dip"], f["rake"], f["length_km"], f["width_km"], f["lon"], f["lat"],
                                f["depth_km"], f["slip_m"]))
    lines.append(f"Receiver_Horizontal_Profile: {PROFILE_DEPTH_KM:.3f} {RECEIVER[0]:.3f} {RECEIVER[1]:.3f} "
                 f"{RECEIVER[2]:.3f} {LON0:.4f} {LAT0:.4f} 20.0 20.0 0.5")
    return "\n".join(lines) + "\n"


def run_esp(name, input_text):
    """Write the input and configuration files of run `name` and run Elastic_stresses_py on them (one thread)."""
    inp = ESP_DIR / f"{name}.intxt"
    inp.write_text(input_text, encoding="utf-8")
    cfg = ESP_DIR / f"{name}_config.txt"
    cfg.write_text("[io-config]\n"
                   f"exp_name = {name}\ninput_file = {inp.resolve()}\noutput_dir = {ESP_DIR.resolve()}\n"
                   f"plot_stress = 1\nplot_grd_disp = 0\nstrain_file = {(ESP_DIR / 'points.txt').resolve()}\n\n"
                   "[compute-config]\nstrike_num_receivers = 1\ndip_num_receivers = 1\n"
                   f"mu = {SHEAR_MODULUS_PA:g}\nlame1 = {LAME1_PA:g}\nB = {SKEMPTON:g}\n", encoding="utf-8")
    env = dict(os.environ, OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1", MPLBACKEND="Agg")
    code = ("import sys; from elastic_stresses_py.PyCoulomb.bin.elastic_stresses_driver import main; "
            "sys.argv = ['elastic_stresses_driver', sys.argv[1]]; main()")
    done = subprocess.run(["nice", "-n", "10", sys.executable, "-c", code, str(cfg)], env=env, capture_output=True,
                          text=True, timeout=600)
    if done.returncode != 0:
        raise RuntimeError(f"Elastic_stresses_py run {name} failed:\n{done.stderr[-2000:]}")
    return ESP_DIR / name


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


runs = {"first": run_esp("first", esp_input(FRICTION)), "rerun": run_esp("rerun", esp_input(FRICTION)),
        "changed": run_esp("changed", esp_input(CHANGED_FRICTION))}
fingerprints = {key: {name: sha256(folder / name) for name in ("ll_stresses.txt", "horizontal_profile_stresses.txt")}
                for key, folder in runs.items()}
_, _, _, tau_first = read_stresses(runs["first"] / "ll_stresses.txt")
esp_cfs = coulomb_change(tau_first, *RECEIVER, FRICTION, SKEMPTON)[2]
mine_cfs = dcfs_at(pts_g["east_km"].to_numpy(), pts_g["north_km"].to_numpy(), pts_g["depth_km"].to_numpy())
esp_vs_pyrocko = float(np.abs(mine_cfs - esp_cfs).max() / np.abs(esp_cfs).max())
sensitivity = []
for rec in (RECEIVER, tuple(AUX), (FAULTS["fore"]["strike"], FAULTS["fore"]["dip"], FAULTS["fore"]["rake"])):
    for fr in (0.0, 0.4, 0.8):
        c = np.asarray(coulomb_change(tau_first, *rec, fr, SKEMPTON)[2])
        sensitivity.append({"receiver": [float(v) for v in rec], "friction": fr, "n": int(c.size),
                            "fraction": float(np.mean(c > 0.0))})
center = (FAULTS["main"]["east_km"], FAULTS["main"]["north_km"])
test = rotation_test(dcfs_at, pts_g["east_km"].to_numpy(), pts_g["north_km"].to_numpy(), pts_g["depth_km"].to_numpy(),
                     center[0], center[1], N_ROTATIONS, np.random.default_rng(SEED))
gyeongju_report = {
    "read_stresses": read_stresses, "coulomb_change": coulomb_change, "rotation_test": rotation_test,
    "okada_stress": okada_stress, "dcfs_at": dcfs_at, "faults": FAULTS, "elastic_pa": [LAME1_PA, SHEAR_MODULUS_PA],
    "runs": {k: str(v) for k, v in runs.items()}, "sha256": fingerprints,
    "changed_friction": CHANGED_FRICTION, "points_csv": str(points_csv), "receiver": list(RECEIVER),
    "friction": FRICTION, "skempton": SKEMPTON, "sensitivity": sensitivity,
    "null": {"seed": SEED, "n_rotations": N_ROTATIONS, "center_km": list(center), "observed": test["observed"],
             "null_mean": float(np.mean(test["null"])), "p_value": test["p_value"]},
    "esp_vs_pyrocko": esp_vs_pyrocko}
print(f"three runs written under {ESP_DIR}; {len(pts_g)} points; the answer gyeongju_report is built")

In [ ]:
st.check("M54-T6", answer=gyeongju_report)

Your results: the fingerprints of the three runs, the two programs compared, the fraction of aftershocks in positive
areas for three receivers and three frictions, and the rotation test.

In [ ]:
for key, files in fingerprints.items():
    print(f"{key:8s}", "  ".join(f"{name}: {digest[:12]}..." for name, digest in files.items()))
print(f"first and rerun identical: {fingerprints['first'] == fingerprints['rerun']}; changed run, stresses identical: "
      f"{fingerprints['changed']['ll_stresses.txt'] == fingerprints['first']['ll_stresses.txt']}, map identical: "
      f"{fingerprints['changed']['horizontal_profile_stresses.txt'] == fingerprints['first']['horizontal_profile_stresses.txt']}")
print(f"Elastic_stresses_py against your pyrocko route at {len(pts_g)} points: largest |difference| / largest |value| "
      f"= {esp_vs_pyrocko:.2e}; same sign at {np.mean(np.sign(mine_cfs) == np.sign(esp_cfs)):.3f} of the points")
print(pd.DataFrame(sensitivity).assign(receiver=lambda d: d["receiver"].map(tuple)).round(3).to_string(index=False))
print(f"rotation test ({N_ROTATIONS} turns about the mainshock centre): observed {test['observed']:.3f}, null mean "
      f"{np.mean(test['null']):.3f} (5-95 %: {np.percentile(test['null'], 5):.3f}-{np.percentile(test['null'], 95):.3f}),"
      f" p {test['p_value']:.3f}")
print(f"points: {len(pts_g)} aftershocks beyond {EXCLUDE_KM} km of both rectangles, at "
      f"{len(pts_g.drop_duplicates(['lon', 'lat', 'depth_km']))} distinct catalog positions")
prof = np.loadtxt(runs["first"] / "horizontal_profile_stresses.txt", ndmin=2)
prof_c = np.loadtxt(runs["changed"] / "horizontal_profile_stresses.txt", ndmin=2)
print(f"map at {PROFILE_DEPTH_KM} km: share of positive Coulomb change {np.mean(prof[:, 7] > 0):.3f} (friction "
      f"{FRICTION}), {np.mean(prof_c[:, 7] > 0):.3f} (friction {CHANGED_FRICTION})")

**One parameter change, viewed spatially.** The two production maps and their difference share color limits.
Dotted contours mark zero, stars mark source epicenters, and points mark the aftershocks used. The maps are
evaluated at one fixed profile depth; the test evaluates each aftershock at its own catalog depth. Your
near-fault exclusion masks unresolved source neighborhoods in all three maps.

**Interpret:** which boundaries shift with friction? Why can a point's map color differ from its individual
computed stress? Use the rotation test and sensitivity comparisons before attributing a pattern to triggering.

In [ ]:
n_side = int(round(np.sqrt(len(prof))))
profile_east, profile_north = to_local(prof[:, 0], prof[:, 1], LON0, LAT0)
profile_valid = np.isfinite(dcfs_at(profile_east, profile_north, np.full(len(prof), PROFILE_DEPTH_KM)))
fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
for ax, values, title in zip(axes, (prof[:, 7], prof_c[:, 7], prof_c[:, 7] - prof[:, 7]),
        (f"Friction {FRICTION:g}", f"Friction {CHANGED_FRICTION:g}", "Changed minus first")):
    field = np.where(profile_valid, values, np.nan).reshape(n_side, n_side)
    px, py = prof[:, 0].reshape(n_side, n_side), prof[:, 1].reshape(n_side, n_side)
    mesh = ax.pcolormesh(px, py, field, cmap="RdBu_r", vmin=-100, vmax=100, shading="auto")
    zero_boundary(ax, px, py, field)
    ax.scatter(pts_g["lon"], pts_g["lat"], s=4, color="k", alpha=0.5)
    ax.scatter([g_fore["longitude"], g_main["longitude"]], [g_fore["latitude"], g_main["latitude"]],
               marker="*", color="gold", edgecolors="k", s=80)
    ax.set(xlabel="Longitude (°E)", ylabel="Latitude (°N)", title=title, aspect=1 / np.cos(np.radians(LAT0)))
fig.colorbar(mesh, ax=axes, label="Stress change (kPa); colors saturate at ±100", extend="both", shrink=0.8)
show_figure(fig, "M54_gyeongju_friction_maps.png")



**Read the spatial result together with its alternatives.** The histogram is your rotation-null distribution
for the selected Gyeongju aftershocks. The right panel changes receiver orientation and friction while keeping
the source stress tensors and point sample fixed. A large positive fraction can be sensitive to those choices;
neither panel is a direct measurement of causation.

**Interpret:** is the observed fraction unusual under this particular null? How does that comparison change
the claim you would make after seeing the sensitivity to receiver assumptions?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.7), constrained_layout=True)
axes[0].hist(test["null"], bins=np.linspace(0, 1, 31), color="0.65", edgecolor="white")
axes[0].axvline(test["observed"], color="tab:red", lw=2, label=f"Observed; p={test['p_value']:.3f}")
axes[0].set(xlabel="Fraction in positive modeled stress", ylabel="Number of rotations", title="Your declared rotation test")
axes[0].legend(fontsize=8)
visual_sensitivity = pd.DataFrame(sensitivity)
visual_sensitivity["receiver"] = visual_sensitivity["receiver"].map(tuple)
for receiver, part in visual_sensitivity.groupby("receiver", sort=False):
    axes[1].plot(part["friction"], part["fraction"], "o-", label="/".join(f"{v:g}" for v in receiver))
axes[1].set(xlabel="Friction coefficient", ylabel="Positive fraction of the same aftershocks", ylim=(0, 1),
            title="Receiver strike/dip/rake sensitivity")
axes[1].legend(fontsize=7, title="Receiver (°)", title_fontsize=8)
show_figure(fig, "M54_gyeongju_null_and_sensitivity.png")

**Explain (3-5 sentences):** Read your printed fingerprints: what did the unchanged rerun show, and which file
changed when only the friction changed, and why? How closely did the two programs agree? Read your sensitivity rows
and your rotation-test row: is the observed fraction above the chance level of the turned clouds, what chance level
did you measure, and how much do receiver and friction move the fraction? Compare your rows for the mainshock plane
and its auxiliary plane at friction 0: what do the two nodal planes of one double couple share that explains what
you see? Say what this simple model (uniform
square faults at the KMA hypocentres, one stress drop, receivers like the mainshock, catalog positions on a 1-km
step) lets you claim about triggering at Gyeongju and what it does not; if a number moved in a way the notebook does
not explain, say so plainly.

## Part 7 · Suncheon: what could a Coulomb test resolve there? (about 20 min)

The cell below prints what the KMA catalog says about the earthquakes near Suncheon, and the fault size of the
largest one if its ML were a moment magnitude (an assumption for small Korean earthquakes, good to a few tenths),
at `STRESS_DROP_MPA`.

In [ ]:
suncheon = st.data.sequence("suncheon")
big = suncheon.sort_values("ml", ascending=False).iloc[0]
side_sc, slip_sc = square_fault(magnitude_moment(mw=float(big["ml"])), STRESS_DROP_MPA * 1e6, SHEAR_MODULUS_PA)
years = suncheon["origin_utc"].dt.year
print(f"{len(suncheon)} earthquakes from {years.min()} to {years.max()}, ML {suncheon['ml'].min()}-{suncheon['ml'].max()};"
      f" {int(suncheon['depth_km'].isna().sum())} without a depth; most in one year: {years.value_counts().max()}")
print(f"largest: ML {big['ml']} on {big['origin_utc']:%Y-%m-%d}; as Mw, a square of side {1000 * side_sc:.0f} m "
      f"and slip {1000 * slip_sc:.1f} mm")
print(f"catalog step in latitude: {np.diff(np.unique(np.round(suncheon['latitude'], 6))).min():.3f} degrees")
sc_ring = np.column_stack([np.zeros(1), np.full(1, 2.0), np.full(1, 10.0)])
sc_fault = {"strike": 0.0, "dip": 90.0, "rake": 180.0, "length_km": side_sc, "width_km": side_sc, "east_km": 0.0,
            "north_km": 0.0, "depth_km": 10.0, "slip_m": slip_sc}
sc_cfs = coulomb_change(okada_stress(sc_fault, sc_ring, LAME1_PA, SHEAR_MODULUS_PA), 0.0, 90.0, 180.0, FRICTION)[2]
print(f"Coulomb change 2 km from the centre of such a fault, along its strike (vertical strike-slip, receivers "
      f"parallel): {sc_cfs[0]:.3f} kPa")



**Compare the scales before interpreting a stress map.** These are three lengths with different meanings:
the modeled source size, a rounding step in the shipped catalog, and this notebook's illustrative near-fault
exclusion distance. The rounding step is not a location error bar, and the chosen exclusion is not a physical
boundary for triggering. Their comparison shows why the resolution of observations matters for a small source.

**Interpret:** what observation would most improve the test: finer map pixels, more accurate relative locations,
better source dimensions, or a constrained receiver mechanism? Explain which uncertainty each would address.

In [ ]:
visual_catalog_step = np.diff(np.unique(np.round(suncheon["latitude"], 6))).min() * 111.2 * 1000
fig, ax = plt.subplots(figsize=(8, 3.2), constrained_layout=True)
visual_lengths = [side_sc * 1000, visual_catalog_step, EXCLUDE_KM * 1000]
visual_labels = ["Modeled square side", "Catalog latitude rounding step", "Illustrative near-fault exclusion"]
ax.scatter(visual_lengths, visual_labels, c=["tab:blue", "0.5", "tab:orange"], s=65)
for label, length in zip(visual_labels, visual_lengths):
    ax.annotate(f"{length:,.0f} m", (length, label), xytext=(8, 0), textcoords="offset points", va="center")
ax.set_xlim(min(visual_lengths) / 1.6, max(visual_lengths) * 2.8)
ax.grid(axis="x", alpha=0.25)
ax.set(xscale="log", xlabel="Length (m; logarithmic scale)", title="Suncheon: source scale and observing limits")
ax.invert_yaxis()
show_figure(fig, "M54_suncheon_scales.png")

**Reflect (not checked, 3-5 sentences):** From the printed lines: how large is the largest Suncheon fault compared
with the catalog's location step and with the 2-km exclusion of Part 6? Is there a mainshock with aftershocks in
time, or a slow scatter of small events? Compare the printed stress change 2 km away with the tides. From these
numbers, could a Coulomb test there say anything? If not, what would you need (relocations, mechanisms, a larger
event) before it could?

## What you did and what comes next

You resolved stress on faults exactly as a production program does, wrote one fault for two programs with
different conventions and showed they agree, sized faults from moment and stress drop, compared stress changes with
the tides, ran Elastic_stresses_py three times with fingerprints, and tested aftershock locations against a rotation
null whose chance level you measured. Module M60 brings the pieces of the course together on one sequence.

Credits: Elastic_stresses_py by Kathryn Materna (Materna and Wong 2023; https://github.com/kmaterna/Elastic_stresses_py)
with cutde (Nikkhoo and Walter 2015) and tectonic-utils; pyrocko (Heimann et al. 2017); moment tensors from the USGS
(ComCat) and the Global CMT project through the ISC Bulletin; KIGAM mechanism of Sangju from KIGAM (2021), report
KEMS-2019-062; KMA catalog (KOGL Type 1); ObsPy (Beyreuther et al. 2010; Krischer et al. 2015).